# 1. Loading the Dataset and Handling Missing Values


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.spatial.distance import pdist, squareform, euclidean, cosine, jaccard
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances



pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
df = pd.read_csv('Lab2_titanic.csv')
print("Dataset loaded. Shape:", df.shape)
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'Lab2_titanic.csv'

In [ ]:
from IPython.display import display

print("Dataset shape:", df.shape)
print("First 100 rows:")
display(df.head(100))
print("\nData types:")
display(df.dtypes)
print("\nMissing values per column:")
display(df.isnull().sum())


Dataset shape: (891, 12)
First 100 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0,3,"Shorney, Mr. Charles Joseph",male,NaN,0,0,374910,8.0500,NaN,S
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C
97,98,1,1,"Greenfield, Mr. William Bertram",male,23.0,0,1,PC 17759,63.3583,D10 D12,C
98,99,1,2,"Doling, Mrs. John T (Ada Julia Bone)",female,34.0,0,1,231919,23.0000,NaN,S



Data types:


PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


Missing values per column:


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [ ]:
print("Missing values in each column:")
missing_values = df.isnull().sum()
print(missing_values)

print("\nPercentage of missing values:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percentage.values
})
print(missing_df[missing_df['Missing Count'] > 0])

Missing values in each column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Percentage of missing values:
      Column  Missing Count  Missing Percentage
5        Age            177           19.865320
10     Cabin            687           77.104377
11  Embarked              2            0.224467


In [ ]:
df_processed = df.copy()

print("Original dataset shape:", df_processed.shape)
print("Missing values before processing:")
print(df_processed.isnull().sum())

Original dataset shape: (891, 12)
Missing values before processing:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


### 3.1 Handling Age Column

In [ ]:
titles = df_processed['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
print("Unique titles:", titles.unique())


title_mapping = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mrs": "Mrs",
    "Master": "Master",
    "Dr": "Officer",
    "Rev": "Officer",
    "Col": "Officer",
    "Major": "Officer",
    "Mlle": "Miss",
    "Countess": "Royalty",
    "Ms": "Miss",
    "Lady": "Royalty",
    "Jonkheer": "Royalty",
    "Don": "Royalty",
    "Dona": "Royalty",
    "Mme": "Mrs",
    "Capt": "Officer",
    "Sir": "Royalty"
}

mapped_titles = titles.map(title_mapping)
print("\nGrouped titles:", mapped_titles.unique())

age_medians = df_processed.groupby([mapped_titles, df_processed['Pclass']])['Age'].median()
print("\nMedian ages by title and class:")
print(age_medians)

def fill_age(row):
    if pd.isna(row['Age']):
        title = titles[row.name]
        mapped_title = mapped_titles[row.name]
        try:
            return age_medians[mapped_title, row['Pclass']]
        except KeyError:
            title_median = df_processed.loc[mapped_titles == mapped_title, 'Age'].median()
            if pd.isna(title_median):
                return df_processed['Age'].median()
            return title_median
    return row['Age']

# display(df_processed.head(100))

df_processed['Age'] = df_processed.apply(fill_age, axis=1)
display(df_processed.head(100))


Unique titles: ['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'Countess' 'Jonkheer']

Grouped titles: ['Mr' 'Mrs' 'Miss' 'Master' 'Royalty' 'Officer']

Median ages by title and class:
Name     Pclass
Master   1          4.0
         2          1.0
         3          4.0
Miss     1         30.0
         2         24.0
         3         18.0
Mr       1         40.0
         2         31.0
         3         26.0
Mrs      1         40.0
         2         32.0
         3         31.0
Officer  1         50.0
         2         46.5
Royalty  1         40.0
Name: Age, dtype: float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0,3,"Shorney, Mr. Charles Joseph",male,26.0,0,0,374910,8.0500,NaN,S
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C
97,98,1,1,"Greenfield, Mr. William Bertram",male,23.0,0,1,PC 17759,63.3583,D10 D12,C
98,99,1,2,"Doling, Mrs. John T (Ada Julia Bone)",female,34.0,0,1,231919,23.0000,NaN,S


In [ ]:
print("Missing values After processing:")
print(df_processed.isnull().sum())

Missing values After processing:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


### 3.2 Handling Embarked Column


In [ ]:
print("Embarked value counts:")
print(df_processed['Embarked'].value_counts())


most_common_embarked = df_processed['Embarked'].mode()[0]
print(f"\nMost common embarked port: {most_common_embarked}")

df_processed.fillna({'Embarked': most_common_embarked}, inplace=True)

print(f"\nEmbarked missing values after filling: {df_processed['Embarked'].isnull().sum()}")

Embarked value counts:
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

Most common embarked port: S

Embarked missing values after filling: 0


### 3.3 Handling Cabin Column

In [ ]:
print(f"Cabin missing values: {df_processed['Cabin'].isnull().sum()}")
print(f"Cabin missing percentage: {(df_processed['Cabin'].isnull().sum() / len(df_processed)) * 100:.2f}%")

df_processed.drop('Cabin', axis=1, inplace=True)

print(f"\nDataset shape after dropping Cabin: {df_processed.shape}")

Cabin missing values: 687
Cabin missing percentage: 77.10%

Dataset shape after dropping Cabin: (891, 11)


In [ ]:
print("Final missing values in processed dataset:")
final_missing = df_processed.isnull().sum()
print(final_missing[final_missing > 0])

print(f"\nTotal missing values: {df_processed.isnull().sum().sum()}")
print(f"Dataset shape: {df_processed.shape}")
print(f"Columns: {list(df_processed.columns)}")

Final missing values in processed dataset:
Series([], dtype: int64)

Total missing values: 0
Dataset shape: (891, 11)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


# 2.Dealing with categorical data: Label encoding and one-hot encoding of categorical variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in ['Sex', 'Embarked']:
    le = LabelEncoder()
    df_processed[col + '_LE'] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    df_processed.drop(col, axis=1, inplace=True)


df_processed = pd.get_dummies(df_processed, columns=['Pclass'], prefix=['Pclass'])

print(df_processed.head())

   PassengerId  Survived                                               Name  \
0            1         0                            Braund, Mr. Owen Harris   
1            2         1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3         1                             Heikkinen, Miss. Laina   
3            4         1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4            5         0                           Allen, Mr. William Henry   

    Age  SibSp  Parch            Ticket     Fare  Sex_LE  Embarked_LE  \
0  22.0      1      0         A/5 21171   7.2500       1            2   
1  38.0      1      0          PC 17599  71.2833       0            0   
2  26.0      0      0  STON/O2. 3101282   7.9250       0            2   
3  35.0      1      0            113803  53.1000       0            2   
4  35.0      0      0            373450   8.0500       1            2   

   Pclass_1  Pclass_2  Pclass_3  
0     False     False      True  
1      True     Fa

# 3. Scaling the features: Perform min-max normalisation and Z-score standardisation.

In [ ]:
numerical_cols = ['Age', 'SibSp', 'Parch', 'Fare']

# Create copies for MinMax scaling
df_minmax = df_processed.copy()
minmax_scaler = MinMaxScaler()
df_minmax[numerical_cols] = minmax_scaler.fit_transform(df_minmax[numerical_cols])

# Create copies for Z-score scaling
df_zscore = df_processed.copy()
zscore_scaler = StandardScaler()
df_zscore[numerical_cols] = zscore_scaler.fit_transform(df_zscore[numerical_cols])

print("MinMax scaled dataset shape:", df_minmax.shape)
print("\nFirst few rows of MinMax scaled data:")
display(df_minmax.head())

print("\nZ-score scaled dataset shape:", df_zscore.shape)
print("\nFirst few rows of Z-score scaled data:")
display(df_zscore.head())

MinMax scaled dataset shape: (891, 13)

First few rows of MinMax scaled data:


,PassengerId,Survived,Name,Age,SibSp,Parch,Ticket,Fare,Sex_LE,Embarked_LE,Pclass_1,Pclass_2,Pclass_3
0,1,0,"Braund, Mr. Owen Harris",0.271174,0.125,0.0,A/5 21171,0.014151,1,2,False,False,True
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0.472229,0.125,0.0,PC 17599,0.139136,0,0,True,False,False
2,3,1,"Heikkinen, Miss. Laina",0.321438,0.000,0.0,STON/O2. 3101282,0.015469,0,2,False,False,True
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0.434531,0.125,0.0,113803,0.103644,0,2,True,False,False
4,5,0,"Allen, Mr. William Henry",0.434531,0.000,0.0,373450,0.015713,1,2,False,False,True



Z-score scaled dataset shape: (891, 13)

First few rows of Z-score scaled data:


,PassengerId,Survived,Name,Age,SibSp,Parch,Ticket,Fare,Sex_LE,Embarked_LE,Pclass_1,Pclass_2,Pclass_3
0,1,0,"Braund, Mr. Owen Harris",-0.529231,0.432793,-0.473674,A/5 21171,-0.502445,1,2,False,False,True
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0.657200,0.432793,-0.473674,PC 17599,0.786845,0,0,True,False,False
2,3,1,"Heikkinen, Miss. Laina",-0.232623,-0.474545,-0.473674,STON/O2. 3101282,-0.488854,0,2,False,False,True
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0.434744,0.432793,-0.473674,113803,0.420730,0,2,True,False,False
4,5,0,"Allen, Mr. William Henry",0.434744,-0.474545,-0.473674,373450,-0.486337,1,2,False,False,True


# 4. Similarity and Dissimilarity Measures:
1. Pearson’s Correlation
JULIA RAHMAN 2
2. Cosine Similarity
3. Jaccard Similarity
4. Euclidean Distance


In [ ]:
numerical_features = ['Age', 'SibSp', 'Parch', 'Fare', 'Sex_LE', 'Embarked_LE', 'Pclass_1', 'Pclass_2', 'Pclass_3']
data_for_analysis = df_processed[numerical_features].astype(float)

print("1. PEARSON'S CORRELATION MATRIX:")
correlation_matrix = data_for_analysis.corr()
print(correlation_matrix)


1. PEARSON'S CORRELATION MATRIX:
                  Age     SibSp     Parch      Fare    Sex_LE  Embarked_LE  \
Age          1.000000 -0.270821 -0.184576  0.121191  0.103141    -0.010490   
SibSp       -0.270821  1.000000  0.414838  0.159651 -0.114631     0.068230   
Parch       -0.184576  0.414838  1.000000  0.216225 -0.245489     0.039798   
Fare         0.121191  0.159651  0.216225  1.000000 -0.182333    -0.224719   
Sex_LE       0.103141 -0.114631 -0.245489 -0.182333  1.000000     0.108262   
Embarked_LE -0.010490  0.068230  0.039798 -0.224719  0.108262     1.000000   
Pclass_1     0.391852 -0.054582 -0.017633  0.591711 -0.098013    -0.237965   
Pclass_2     0.027673 -0.055932 -0.000734 -0.118557 -0.064746     0.169245   
Pclass_3    -0.360143  0.092548  0.015790 -0.413333  0.137143     0.067291   

             Pclass_1  Pclass_2  Pclass_3  
Age          0.391852  0.027673 -0.360143  
SibSp       -0.054582 -0.055932  0.092548  
Parch       -0.017633 -0.000734  0.015790  
Fare      

In [ ]:
print("\n2. COSINE SIMILARITY (First 10 passengers):")
sample_data = data_for_analysis.head(10)
cosine_sim_matrix = cosine_similarity(sample_data)
print(f"Cosine Similarity Matrix Shape: {cosine_sim_matrix.shape}")
cosine_sim_df = pd.DataFrame(cosine_sim_matrix, 
                            index=[f"Passenger_{i}" for i in range(10)],
                            columns=[f"Passenger_{i}" for i in range(10)])
print(cosine_sim_df)



2. COSINE SIMILARITY (First 10 passengers):
Cosine Similarity Matrix Shape: (10, 10)
             Passenger_0  Passenger_1  Passenger_2  Passenger_3  Passenger_4  \
Passenger_0     1.000000     0.718727     0.997809     0.781742     0.994170   
Passenger_1     0.718727     1.000000     0.704778     0.995177     0.654624   
Passenger_2     0.997809     0.704778     1.000000     0.769010     0.996995   
Passenger_3     0.781742     0.995177     0.769010     1.000000     0.723020   
Passenger_4     0.994170     0.654624     0.996995     0.723020     1.000000   
Passenger_5     0.997808     0.718783     0.998481     0.780645     0.995850   
Passenger_6     0.898361     0.950052     0.890343     0.975084     0.857557   
Passenger_7     0.409709     0.908477     0.381262     0.872307     0.316754   
Passenger_8     0.993029     0.767136     0.993168     0.824270     0.983706   
Passenger_9     0.680857     0.997854     0.665197     0.988052     0.612551   

             Passenger_5  Passeng

In [ ]:
print("\n3. JACCARD SIMILARITY (Binary features only - First 10 passengers):")
binary_features = ['Pclass_1', 'Pclass_2', 'Pclass_3']
binary_data = df_processed[binary_features].head(10).astype(int)

jaccard_distances = pdist(binary_data, metric='jaccard')
jaccard_similarity_matrix = 1 - squareform(jaccard_distances)
jaccard_sim_df = pd.DataFrame(jaccard_similarity_matrix,
                             index=[f"Passenger_{i}" for i in range(10)],
                             columns=[f"Passenger_{i}" for i in range(10)])
print(jaccard_sim_df)



3. JACCARD SIMILARITY (Binary features only - First 10 passengers):
             Passenger_0  Passenger_1  Passenger_2  Passenger_3  Passenger_4  \
Passenger_0          1.0          0.0          1.0          0.0          1.0   
Passenger_1          0.0          1.0          0.0          1.0          0.0   
Passenger_2          1.0          0.0          1.0          0.0          1.0   
Passenger_3          0.0          1.0          0.0          1.0          0.0   
Passenger_4          1.0          0.0          1.0          0.0          1.0   
Passenger_5          1.0          0.0          1.0          0.0          1.0   
Passenger_6          0.0          1.0          0.0          1.0          0.0   
Passenger_7          1.0          0.0          1.0          0.0          1.0   
Passenger_8          1.0          0.0          1.0          0.0          1.0   
Passenger_9          0.0          0.0          0.0          0.0          0.0   

             Passenger_5  Passenger_6  Passenger_7

In [ ]:
print("\n4. EUCLIDEAN DISTANCE MATRIX (First 10 passengers):")
euclidean_dist_matrix = euclidean_distances(sample_data)
euclidean_dist_df = pd.DataFrame(euclidean_dist_matrix,
                                index=[f"Passenger_{i}" for i in range(10)],
                                columns=[f"Passenger_{i}" for i in range(10)])
print(euclidean_dist_df)



4. EUCLIDEAN DISTANCE MATRIX (First 10 passengers):
             Passenger_0  Passenger_1  Passenger_2  Passenger_3  Passenger_4  \
Passenger_0     0.000000    66.055004     4.296001    47.688809    13.062925   
Passenger_1    66.055004     0.000000    64.538935    18.537324    63.367580   
Passenger_2     4.296001    64.538935     0.000000    46.095343     9.056248   
Passenger_3    47.688809    18.537324    46.095343     0.000000    45.094373   
Passenger_4    13.062925    63.367580     9.056248    45.094373     0.000000   
Passenger_5     4.411348    63.999849     1.511426    45.594752     9.064585   
Passenger_6    54.929729    25.281762    52.129684    19.092706    47.775885   
Passenger_7    24.415786    61.877891    27.566692    46.071690    35.618122   
Passenger_8     6.788226    61.237427     3.910651    42.804251     8.860403   
Passenger_9    24.326712    47.712369    25.326596    31.262502    30.560033   

             Passenger_5  Passenger_6  Passenger_7  Passenger_8  P